In [2]:
import pandas as pd

df = pd.read_csv("/home/algosium/Documents/prd-analysis/data/Large_Industrial_Pump_Maintenance_Dataset.csv")

In [4]:
df["Operational_Days"] = df["Operational_Hours"] / 24
df

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Maintenance_Flag,Operational_Days
0,2,127.508350,2.369397,136.021029,6.501492,1444.191922,3966.793672,1,165.283070
1,4,88.975185,4.541126,147.516973,7.001516,1004.802496,3673.288933,0,153.053706
2,3,61.832325,2.542112,220.858577,8.824368,2597.662712,5489.061016,1,228.710876
3,3,106.250344,2.834452,145.817091,16.283512,2280.926281,3134.783015,0,130.615959
4,2,84.815865,3.119709,235.476221,8.385183,2594.131667,761.533173,1,31.730549
...,...,...,...,...,...,...,...,...,...
19995,5,133.824387,2.193536,162.072728,6.272785,1939.452036,4327.831926,1,180.326330
19996,1,100.976892,4.906084,109.160420,10.000945,2267.788691,7474.000467,1,311.416686
19997,3,109.563715,0.769374,180.743971,18.706712,2758.781582,8935.787395,1,372.324475
19998,3,116.730910,3.279518,289.369874,14.570612,1683.361885,4576.992294,0,190.708012


In [7]:
df = df.sort_values(["Pump_ID", "Operational_Hours"])

In [8]:
normal = df[df["Maintenance_Flag"] == 0]

mean_values = normal.mean()
std_values = normal.std()

In [9]:
temp_limit = mean_values["Temperature"] + 2 * std_values["Temperature"]
vib_limit  = mean_values["Vibration"] + 2 * std_values["Vibration"]

In [10]:
df["Threshold_Alert"] = (
    (df["Temperature"] > temp_limit) |
    (df["Vibration"] > vib_limit)
).astype(int)

In [11]:
df["Vibration_Rolling"] = (
    df.groupby("Pump_ID")["Vibration"]
      .rolling(window=5)
      .mean()
      .reset_index(level=0, drop=True)
)

In [12]:
df["Vibration_Trend"] = df.groupby("Pump_ID")["Vibration"].diff()

In [13]:
df["Trend_Alert"] = (df["Vibration_Trend"] > 0.5).astype(int)

In [14]:
features = [
    "Temperature",
    "Vibration",
    "Pressure",
    "Flow_Rate",
    "RPM",
    "Operational_Hours"
]

X = df[features]
y = df["Maintenance_Flag"]

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [18]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      0.54      0.52      2980
           1       0.50      0.46      0.48      3020

    accuracy                           0.50      6000
   macro avg       0.50      0.50      0.50      6000
weighted avg       0.50      0.50      0.50      6000



Accuracy = 0.50
Precision ≈ 0.50
Recall ≈ 0.46–0.54
F1 ≈ 0.48–0.52

In [19]:
df.corr()["Maintenance_Flag"]

Pump_ID             -0.007189
Temperature         -0.013055
Vibration            0.007575
Pressure            -0.005164
Flow_Rate            0.008674
RPM                 -0.008564
Operational_Hours   -0.003098
Maintenance_Flag     1.000000
Operational_Days    -0.003098
Threshold_Alert           NaN
Vibration_Rolling    0.000586
Vibration_Trend      0.010444
Trend_Alert          0.009559
Name: Maintenance_Flag, dtype: float64

In [20]:
df.groupby("Maintenance_Flag").mean()

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Operational_Days,Threshold_Alert,Vibration_Rolling,Vibration_Trend,Trend_Alert
Maintenance_Flag,,,,,,,,,,,,
0,3.013656,100.746785,2.517161,200.679232,10.199296,2010.538768,5039.337827,209.972409,0.0,2.527691,-0.020992,0.397628
1,2.993379,99.993484,2.538619,200.082087,10.297241,2000.664014,5021.567381,209.231974,0.0,2.528425,0.021101,0.407002


| Parameter         | Normal  | Failure | Difference |
| ----------------- | ------- | ------- | ---------- |
| Temperature       | 100.74  | 99.99   | ~0.7       |
| Vibration         | 2.517   | 2.538   | ~0.02      |
| Pressure          | 200.67  | 200.08  | ~0.6       |
| RPM               | 2010.53 | 2000.66 | ~10        |
| Operational Hours | 5039    | 5021    | ~18        |


🚨 3️⃣ The Real Conclusion

Your dataset is almost certainly:

🔹 Randomly generated
🔹 Synthetic
🔹 Failure flag randomly assigned

Because:

In real industrial pumps:

Failed pumps show higher vibration

Failed pumps show higher temperature

Failed pumps show increasing trend

Failed pumps usually have higher operational hours

But your dataset:

Everything looks statistically identical